# Starting-QB data checks

In [ ]:
%load_ext autoreload
%autoreload 2

## How often is the schedule's starting QB wrong?

Compares the schedule's QB field with the QB who threw the most passes for that team in each game.
A normal season is 1.5–3% (mostly starters who left injured early); 2024 is about 3× that, and
2026 was ~6% when checked, e.g. SEA weeks 3–6 and PHI week 7.

This audit led to taking starters from play-by-play. Note that "most passes" is fine for an audit
but **leaks** as a model input (it credits a backup who came in mid-game), so the pipeline uses
each team's **first** passer instead; see `features/qb.py`.

In [9]:
import nflreadpy as nfl
import pandas as pd
from nflmodel.data.games import load_games

hist = nfl.load_pbp(list(range(2015, 2026))).select(
    ["game_id", "posteam", "passer_player_id"]).to_pandas()
hist = hist[hist["passer_player_id"].notna()]
top = (hist.groupby(["game_id", "posteam", "passer_player_id"]).size()
       .reset_index(name="n").sort_values("n")
       .groupby(["game_id", "posteam"]).tail(1))

g = load_games()
g = g[g["season"].between(2015, 2025)]
sched = pd.concat([
    g[["game_id", "season", "home_team", "home_qb_id"]].set_axis(["game_id", "season", "team", "qb_id"], axis=1),
    g[["game_id", "season", "away_team", "away_qb_id"]].set_axis(["game_id", "season", "team", "qb_id"], axis=1),
])
check = sched.merge(top, left_on=["game_id", "team"], right_on=["game_id", "posteam"])
check["mismatch"] = check["qb_id"] != check["passer_player_id"]
print(check.groupby("season")["mismatch"].mean().round(3))

season
2015    0.026
2016    0.024
2017    0.028
2018    0.015
2019    0.024
2020    0.019
2021    0.025
2022    0.032
2023    0.032
2024    0.088
2025    0.037
Name: mismatch, dtype: float64


## Replaying the QB check on week 6

Rebuilds the Wednesday view of week 6 (each starter projected as the team's last starter), runs
the depth-chart and injury-report check, and compares it with who actually started.

*Saved output is from the original week 6 replay (October 2026).*

In [1]:
import numpy as np
from nflmodel.data.games import load_games
from nflmodel.features.qb import fill_upcoming_qbs
from nflmodel.qb_check import depth_chart_qbs, qb_injury_statuses, check_qbs

SEASON, WEEK = 2026, 6

# rebuild the "Wednesday" view: week 6 unplayed, so starters are projected (last starter)
g = load_games(include_upcoming=True)
g = g[(g["season"] < SEASON) | ((g["season"] == SEASON) & (g["week"] <= WEEK))].copy()
week6 = (g["season"] == SEASON) & (g["week"] == WEEK)
actual = g.loc[week6, ["home_team", "home_qb_id", "away_team", "away_qb_id"]].copy()  # who really started
g.loc[week6, "result"] = np.nan
projected = fill_upcoming_qbs(g)
this_week = projected[week6]

# 1. what the injury report says for week 6 QBs
injuries = qb_injury_statuses(SEASON, WEEK)
qb1, names = depth_chart_qbs(SEASON)
print({names.get(k, k): v for k, v in injuries.items()})

# 2. run the check exactly as predict.py would
print("\nQB check:")
check_qbs(this_week, qb1, names, injuries, overrides={})

# 3. answer key: where the projection was actually wrong
print("\nActually wrong projections:")
for side in ["home", "away"]:
    wrong = this_week[f"{side}_qb_id"].values != actual[f"{side}_qb_id"].values
    for team, qb, real in zip(this_week[f"{side}_team"][wrong], this_week[f"{side}_qb_id"][wrong],
                              actual[f"{side}_qb_id"][wrong]):
        print(f"  {team}: projected {names.get(qb, qb)}, actually started {names.get(real, real)}")

{'Lamar Jackson': 'Out', 'Caleb Williams': 'Questionable', 'Dillon Gabriel': 'Out', 'Jalen Hurts': 'Out', 'Baker Mayfield': 'Out', 'Marcus Mariota': 'Out'}

QB check:
  CHECK PHI: projected Jalen Hurts is listed OUT
  CHECK PHI: projected Jalen Hurts, depth chart QB1 Tanner McKee
  CHECK SEA: projected Drew Lock, depth chart QB1 Sam Darnold
  CHECK CHI: projected Tyson Bagent, depth chart QB1 Caleb Williams

Actually wrong projections:
  PHI: projected Jalen Hurts, actually started Tanner McKee
  CHI: projected Tyson Bagent, actually started Caleb Williams
